
# IMELE / IM2ELEVATION — Kaggle Inference (fixed)

This version fixes the failure caused by the repository's hard-coded debug reshape:

```python
x_block0.view(-1, 250, 250)
```

That value is never used by the decoder, so the notebook removes **only those two debug lines**
from `models/net.py` while keeping the trained architecture and checkpoint unchanged.

The official IMELE test path uses 440×440 crops and upsamples the final prediction to 440×440.
Large images are therefore tiled at 440×440 with overlap and stitched back together.

In [ ]:
# from IPython.display import display
# from ipywidgets import FileUpload
# from pathlib import Path

# upload = FileUpload(
#     accept="image/*",
#     multiple=False
# )

# display(upload)

In [ ]:
# from pathlib import Path

# for fileinfo in upload.value:
#     filename = fileinfo["name"]
#     output_path = Path("/kaggle/working") / filename

#     with open(output_path, "wb") as f:
#         f.write(fileinfo["content"])

#     print(f"Saved to: {output_path}")

In [ ]:

# 1. Configuration

from pathlib import Path
import sys
import os
import re
import shutil
import importlib
import urllib.request

import torch

WEIGHTS_PATH = Path("/kaggle/working/Block0_skip_model_110.pth.tar")

# Can be a single image OR a directory.
IMAGE_INPUT = Path(
    "/kaggle/working/osiDataset/test_rgbs_base"
)

# IMELE test-time crop size
TILE_SIZE = 440
OVERLAP = 220
BATCH_SIZE = 1

OUTPUT_DIR = Path("/kaggle/working/imele_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

print("Device:", DEVICE)
print("Checkpoint:", WEIGHTS_PATH)
print("Image input:", IMAGE_INPUT)
print("Tile size:", TILE_SIZE)
print("Overlap:", OVERLAP)
print("Output:", OUTPUT_DIR)

In [ ]:

# 2. Validate inputs and discover images

IMAGE_EXTENSIONS = {
    ".jpg", ".jpeg", ".png", ".bmp",
    ".tif", ".tiff", ".webp"
}

if not WEIGHTS_PATH.exists():
    raise FileNotFoundError(f"Checkpoint not found: {WEIGHTS_PATH}")

if not IMAGE_INPUT.exists():
    raise FileNotFoundError(f"Image input not found: {IMAGE_INPUT}")

if IMAGE_INPUT.is_file():
    IMAGE_PATHS = [IMAGE_INPUT]
else:
    IMAGE_PATHS = sorted(
        p for p in IMAGE_INPUT.rglob("*")
        if p.is_file() and p.suffix.lower() in IMAGE_EXTENSIONS
    )

if not IMAGE_PATHS:
    raise FileNotFoundError(f"No images found in {IMAGE_INPUT}")

print(f"Found {len(IMAGE_PATHS)} image(s):")
for p in IMAGE_PATHS:
    print(" ", p)

In [ ]:

# 3. Download a FRESH copy of the original IMELE source and patch net.py robustly
#
# Important: the previous notebook reused /kaggle/working/imele_src if it already
# existed. That could leave an old, unpatched net.py in place. We deliberately
# delete the source directory and download a clean copy every run.

SRC = Path("/kaggle/working/imele_src")
if SRC.exists():
    shutil.rmtree(SRC)

MODELS = SRC / "models"
MODELS.mkdir(parents=True, exist_ok=True)

BASE = "https://raw.githubusercontent.com/speed8928/IMELE/master"

FILES = [
    "models/net.py",
    "models/modules.py",
    "models/senet.py",
    "models/resnet.py",
    "models/densenet.py",
]

for rel in FILES:
    dst = SRC / rel
    dst.parent.mkdir(parents=True, exist_ok=True)
    print("Downloading:", rel)
    urllib.request.urlretrieve(f"{BASE}/{rel}", dst)

(MODELS / "__init__.py").write_text(
    "# IMELE local package\n",
    encoding="utf-8"
)

# ---- Patch net.py with a REGEX, not an exact-space string replacement ----
net_path = MODELS / "net.py"
net_text = net_path.read_text(encoding="utf-8")

# Remove unrelated legacy imports.
net_text = re.sub(r"^\s*import util\s*$\n?", "", net_text, flags=re.MULTILINE)
net_text = re.sub(
    r"^\s*from torchvision import utils\s*$\n?",
    "",
    net_text,
    flags=re.MULTILINE,
)

# Remove the two unused debug statements regardless of whitespace formatting.
net_text, n1 = re.subn(
    r"(?m)^\s*x\s*=\s*x_block0\.view\(\-1\s*,\s*250\s*,\s*250\)\s*\n",
    "",
    net_text,
)

net_text, n2 = re.subn(
    r"(?m)^\s*x\s*=\s*x\.cpu\(\)\.detach\(\)\.numpy\(\)\s*\n",
    "",
    net_text,
)

net_path.write_text(net_text, encoding="utf-8")

# Remove the unused harmonic import from senet.py if present.
senet_path = MODELS / "senet.py"
senet_text = senet_path.read_text(encoding="utf-8")
senet_text = re.sub(
    r"^\s*from harmonic import Harm2d\s*$\n?",
    "",
    senet_text,
    flags=re.MULTILINE,
)
senet_path.write_text(senet_text, encoding="utf-8")

remaining_bad = re.findall(
    r"x_block0\.view\(\-1\s*,\s*250\s*,\s*250\)",
    net_text,
)

print(f"Removed debug reshape lines: {n1}")
print(f"Removed debug numpy-conversion lines: {n2}")
print("Remaining hard-coded 250x250 debug reshape:", len(remaining_bad))

if remaining_bad:
    raise RuntimeError(
        "PATCH FAILED: net.py still contains x_block0.view(-1,250,250)"
    )

print("Patched net.py successfully:", net_path)

In [ ]:

# 4. Import the patched model source
#
# Clear any previously imported 'models' package so the current patched files
# are actually used in this notebook session.

for name in list(sys.modules):
    if name == "models" or name.startswith("models."):
        del sys.modules[name]

importlib.invalidate_caches()

if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from models import modules, net, resnet, densenet, senet

print("Imported patched models package from:")
print(SRC)

In [ ]:

# 5. Reconstruct the exact IMELE model and load the checkpoint

def define_imele_model():
    # Same construction used by the repository's test path.
    original_model = senet.senet154(pretrained=None)

    encoder = modules.E_senet(original_model)

    return net.model(
        encoder,
        num_features=2048,
        block_channel=[256, 512, 1024, 2048],
    )


def load_state_dict_from_checkpoint(path):
    # First try the safe loader requested by modern PyTorch.
    try:
        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=True,
        )
    except Exception:
        ckpt = torch.load(
            path,
            map_location="cpu",
            weights_only=False,
        )

    # Common checkpoint formats.
    if isinstance(ckpt, dict):
        for key in ("state_dict", "model_state_dict", "model", "weights"):
            if key in ckpt and isinstance(ckpt[key], dict):
                ckpt = ckpt[key]
                break

    if not isinstance(ckpt, dict):
        raise TypeError(
            f"Checkpoint did not resolve to a state_dict: {type(ckpt)}"
        )

    state_dict = {}

    for k, v in ckpt.items():
        if not isinstance(k, str):
            continue

        # Remove DataParallel prefix if present.
        k = k[7:] if k.startswith("module.") else k
        state_dict[k] = v

    return state_dict


model = define_imele_model()

state_dict = load_state_dict_from_checkpoint(
    WEIGHTS_PATH
)

missing, unexpected = model.load_state_dict(
    state_dict,
    strict=False
)

print("Checkpoint tensors:", len(state_dict))
print("Missing keys:", len(missing))
print("Unexpected keys:", len(unexpected))

if missing:
    print("First missing keys:", missing[:20])
    raise RuntimeError(
        "Checkpoint is missing required model parameters."
    )

if unexpected:
    print("Unexpected keys:", unexpected[:20])

print("\nCheckpoint successfully loaded.")

model = model.to(DEVICE).eval()

for p in model.parameters():
    p.requires_grad_(False)

print("Model ready on:", DEVICE)

In [ ]:

# 6. Preprocessing

import numpy as np
from PIL import Image

IMAGENET_MEAN = np.array(
    [0.485, 0.456, 0.406],
    dtype=np.float32
)

IMAGENET_STD = np.array(
    [0.229, 0.224, 0.225],
    dtype=np.float32
)


def load_rgb(path):
    return Image.open(path).convert("RGB")


def pil_to_tensor(img):
    arr = np.asarray(
        img.convert("RGB"),
        dtype=np.float32
    ) / 255.0

    arr = (
        arr - IMAGENET_MEAN
    ) / IMAGENET_STD

    arr = np.transpose(
        arr,
        (2, 0, 1)
    )

    return torch.from_numpy(
        arr.copy()
    ).float()


def pad_image(img, tile_size):
    w, h = img.size

    padded_w = max(w, tile_size)
    padded_h = max(h, tile_size)

    if (padded_w, padded_h) == (w, h):
        return img

    canvas = Image.new(
        "RGB",
        (padded_w, padded_h),
        (0, 0, 0)
    )

    canvas.paste(img, (0, 0))

    return canvas


def get_starts(length, tile_size, overlap):
    step = tile_size - overlap

    if step <= 0:
        raise ValueError(
            "OVERLAP must be smaller than TILE_SIZE."
        )

    if length <= tile_size:
        return [0]

    starts = list(
        range(
            0,
            length - tile_size + 1,
            step
        )
    )

    last = length - tile_size

    if starts[-1] != last:
        starts.append(last)

    return starts

In [ ]:

# 7. Inference
#
# Input to IMELE: 440x440
# Raw model output: 220x220
# Official test behavior: interpolate output to 440x440
#
# This cell does NOT change the network architecture.

import torch.nn.functional as F


@torch.inference_mode()
def predict_batch(batch):
    x = torch.stack(
        batch,
        dim=0
    ).to(
        DEVICE,
        non_blocking=True
    )

    y = model(x)

    # Match repository test-time output size.
    y = F.interpolate(
        y,
        size=(TILE_SIZE, TILE_SIZE),
        mode="bilinear",
        align_corners=False
    )

    return y[:, 0].float().cpu().numpy()


def predict_image(path):

    image = load_rgb(path)

    original_w, original_h = image.size

    work_img = pad_image(
        image,
        TILE_SIZE
    )

    W, H = work_img.size

    xs = get_starts(
        W,
        TILE_SIZE,
        OVERLAP
    )

    ys = get_starts(
        H,
        TILE_SIZE,
        OVERLAP
    )

    prediction_sum = np.zeros(
        (H, W),
        dtype=np.float32
    )

    prediction_count = np.zeros(
        (H, W),
        dtype=np.float32
    )

    batch = []
    locations = []

    def flush():

        nonlocal batch
        nonlocal locations

        if not batch:
            return

        predictions = predict_batch(batch)

        for prediction, (x0, y0) in zip(
            predictions,
            locations
        ):

            # prediction is exactly 440x440 here.
            assert prediction.shape == (
                TILE_SIZE,
                TILE_SIZE
            ), (
                f"Unexpected model output: "
                f"{prediction.shape}"
            )

            prediction_sum[
                y0:y0 + TILE_SIZE,
                x0:x0 + TILE_SIZE
            ] += prediction

            prediction_count[
                y0:y0 + TILE_SIZE,
                x0:x0 + TILE_SIZE
            ] += 1.0

        batch = []
        locations = []

    for y0 in ys:

        for x0 in xs:

            tile = work_img.crop(
                (
                    x0,
                    y0,
                    x0 + TILE_SIZE,
                    y0 + TILE_SIZE
                )
            )

            batch.append(
                pil_to_tensor(tile)
            )

            locations.append(
                (x0, y0)
            )

            if len(batch) >= BATCH_SIZE:
                flush()

    flush()

    prediction = (
        prediction_sum /
        np.maximum(
            prediction_count,
            1e-6
        )
    )

    # Remove any padding.
    prediction = prediction[
        :original_h,
        :original_w
    ]

    return image, prediction


print("Inference functions ready.")

In [ ]:

# 8. Save and visualize

import matplotlib.pyplot as plt


def normalize_for_png(arr):

    finite = np.isfinite(arr)

    if not finite.any():
        return np.zeros(
            arr.shape,
            dtype=np.uint8
        )

    values = arr[finite]

    low = np.percentile(
        values,
        1
    )

    high = np.percentile(
        values,
        99
    )

    if high <= low:
        high = low + 1e-6

    out = (
        arr - low
    ) / (
        high - low
    )

    out = np.clip(
        out,
        0,
        1
    )

    return (
        out * 255
    ).astype(np.uint8)


def save_prediction(
    name,
    prediction
):

    stem = Path(name).stem

    npy_path = (
        OUTPUT_DIR /
        f"{stem}_imele_prediction.npy"
    )

    png_path = (
        OUTPUT_DIR /
        f"{stem}_imele_prediction.png"
    )

    np.save(
        npy_path,
        prediction.astype(np.float32)
    )

    Image.fromarray(
        normalize_for_png(prediction),
        mode="L"
    ).save(png_path)

    return npy_path, png_path


def visualize(
    image,
    prediction,
    title
):

    fig, axes = plt.subplots(
        1,
        3,
        figsize=(16, 5)
    )

    axes[0].imshow(image)
    axes[0].set_title(
        "Input aerial image"
    )
    axes[0].axis("off")

    im = axes[1].imshow(
        prediction,
        cmap="turbo"
    )
    axes[1].set_title(
        "IMELE predicted DSM"
    )
    axes[1].axis("off")

    fig.colorbar(
        im,
        ax=axes[1],
        fraction=0.046,
        pad=0.04
    )

    axes[2].imshow(image)
    axes[2].imshow(
        prediction,
        cmap="turbo",
        alpha=0.45
    )
    axes[2].set_title(
        "Overlay"
    )
    axes[2].axis("off")

    fig.suptitle(title)

    plt.tight_layout()
    plt.show()

In [ ]:

# 9. Run inference

results = []

for image_path in IMAGE_PATHS:

    print(
        f"\nRunning: {image_path.name}"
    )

    image, prediction = predict_image(
        image_path
    )

    npy_path, png_path = save_prediction(
        image_path.name,
        prediction
    )

    finite = np.isfinite(prediction)
    values = prediction[finite]

    stats = {
        "image": image_path.name,
        "shape": prediction.shape,
        "min": float(values.min()),
        "max": float(values.max()),
        "mean": float(values.mean()),
        "median": float(np.median(values)),
    }

    results.append(stats)

    print(
        "Prediction shape:",
        prediction.shape
    )

    print(
        f"Min:    {stats['min']:.4f}"
    )

    print(
        f"Max:    {stats['max']:.4f}"
    )

    print(
        f"Mean:   {stats['mean']:.4f}"
    )

    print(
        f"Median: {stats['median']:.4f}"
    )

    print("Raw prediction:", npy_path)
    print("Visualization:", png_path)

    visualize(
        image,
        prediction,
        image_path.name
    )

print("\nInference completed.")

In [ ]:

# 10. Results table

import pandas as pd

results_df = pd.DataFrame(results)
results_df


## What this version fixes

The previous notebook failed because its patch could silently miss the exact formatting of
the two debug statements in `net.py`, while Python could also keep the previously imported
`models` package.

This version:

1. Deletes the old `/kaggle/working/imele_src` every run.
2. Downloads a fresh repository copy.
3. Removes the two problematic debug statements with regex matching.
4. **Verifies that `x_block0.view(-1,250,250)` is actually gone.**
5. Clears `models` from `sys.modules` before importing it again.
6. Keeps the real model input at **440×440**, matching the repository test path.
7. Upsamples the network output to 440×440 before stitching.